# Standalone synthetic oracle-aligned CNN training
Three prespecified horizons; no tuning or test-driven selection. Read-only v3 HDF5. All model and training code lives here. Re-execution refuses to overwrite checkpoints. Separate evaluation begins only after all three checkpoints exist.

In [1]:
from pathlib import Path
import os, json, hashlib, random, time, copy, platform
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np
import h5py
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path(r'C:/Users/choud/Desktop/Minor project/cnn/cnn wave/eew_causal_v3')
OUT=ROOT/'cnn_eew_training'
CFG=dict(seed=42,batch_size=256,max_epochs=50,patience=8,min_delta=1e-4,learning_rate=1e-3,weight_decay=1e-4,gradient_clip=5.,magnitude_loss_weight=1.,huber_delta=1.,detection_threshold=.5)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False
def seed_all(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
def sha(path):
 h=hashlib.sha256()
 with path.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
class EarlyCNN(nn.Module):
 def __init__(self):
  super().__init__()
  self.features=nn.Sequential(nn.Conv1d(1,16,9,stride=2,padding=4),nn.SiLU(),nn.Conv1d(16,32,7,stride=2,padding=3),nn.SiLU(),nn.Conv1d(32,48,5,stride=2,padding=2),nn.SiLU())
  self.shared=nn.Sequential(nn.Linear(96,32),nn.SiLU())
  self.detection=nn.Linear(32,1);self.magnitude=nn.Linear(32,1)
 def forward(self,x):
  # Fixed pointwise transform preserves timing and uses no learned scaler or future samples.
  x=torch.sign(x)*torch.log1p(torch.abs(x))
  z=self.features(x);z=self.shared(torch.cat([z.mean(-1),z.amax(-1)],dim=1))
  return self.detection(z).squeeze(1),self.magnitude(z).squeeze(1)
def load_split(split,h):
 # Only waveform enters forward(); labels/masks are loss arguments, IDs/timing never inputs.
 with h5py.File(ROOT/'dataset'/f'eew_dataset_{split}.h5','r') as f:
  keep=f[f'classification_mask_{h}s'][:].astype(bool)
  reg=f[f'magnitude_mask_{h}s'][:].astype(bool)
  y=f['labels'][:].astype(np.float32);mag=f['magnitude'][:].astype(np.float32)
  # Interface guards, not a repetition of the scientific dataset audit. Stop on failure.
  assert np.all(~reg|keep) and np.all(y[reg]==1) and np.all(mag[y==0]==-1)
  assert np.array_equal(reg,keep&(y==1)) and np.all(keep[y==0])
  x=f[f'windows_{h}s_norm'][:][keep,None,:]
  assert x.shape==(int(keep.sum()),1,100+100*h) and np.isfinite(x).all()
  assert np.isfinite(mag[reg]).all()
  rows=np.flatnonzero(keep)
 return {'x':torch.from_numpy(x),'y':torch.from_numpy(y[keep]),'mag':torch.from_numpy(mag[keep]),'reg':torch.from_numpy(reg[keep]),'rows':rows,'excluded':int((~keep).sum())}
def loader(data,shuffle):
 g=torch.Generator().manual_seed(CFG['seed'])
 return DataLoader(TensorDataset(data['x'],data['y'],data['mag'],data['reg']),batch_size=CFG['batch_size'],shuffle=shuffle,num_workers=0,generator=g)
bce=nn.BCEWithLogitsLoss(reduction='sum');huber=nn.HuberLoss(delta=CFG['huber_delta'],reduction='sum')
def objective(logit,pred,y,mag,reg,center,scale):
 detection=bce(logit,y)
 magnitude=huber(pred[reg],(mag[reg]-center)/scale) if reg.any() else pred.sum()*0.
 return detection/len(y)+CFG['magnitude_loss_weight']*magnitude/reg.sum().clamp_min(1),detection,magnitude
# Verify noise sentinel exclusion and empty-regression batches using an isolated loss probe.
probe_y=torch.tensor([1.,0.]);probe_reg=probe_y.bool();probe_mag=torch.tensor([4.,-1.])
a=objective(torch.zeros(2),torch.zeros(2),probe_y,probe_mag,probe_reg,4.,1.)[0]
poison=probe_mag.clone();poison[~probe_reg]=1e9
assert torch.equal(a,objective(torch.zeros(2),torch.zeros(2),probe_y,poison,probe_reg,4.,1.)[0])
assert torch.isfinite(objective(torch.zeros(2),torch.zeros(2),torch.zeros(2),torch.full((2,),-1.),torch.zeros(2,dtype=torch.bool),4.,1.)[0])
def epoch(model,batches,center,scale,opt=None):
 model.train(opt is not None);ds=ms=0.;n=ne=0
 with torch.set_grad_enabled(opt is not None):
  for x,y,mag,reg in batches:
   x,y,mag,reg=[v.to(DEVICE) for v in [x,y,mag,reg]]
   if opt is not None:opt.zero_grad(set_to_none=True)
   logits,pred=model(x);loss,dl,ml=objective(logits,pred,y,mag,reg,center,scale)
   if not torch.isfinite(loss):raise RuntimeError('Nonfinite training loss; stop')
   if opt is not None:
    loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip']);opt.step()
   ds+=dl.item();ms+=ml.item();n+=len(y);ne+=int(reg.sum())
 return {'detection':ds/n,'magnitude':ms/max(ne,1),'total':ds/n+CFG['magnitude_loss_weight']*ms/max(ne,1)}
print('Device:',str(DEVICE),'; parameters:',sum(p.numel() for p in EarlyCNN().parameters()))


Device: cuda ; parameters: 14674


In [2]:
# Inspect schema once; test values are not accessed before all models are selected.
protected=[ROOT/'dataset'/f'eew_dataset_{s}.h5' for s in ['train','validation','test']]+[ROOT/'codex_final_verification/hdf5_training_readiness.md']
before={str(p):sha(p) for p in protected}
for split,n in [('train',11324),('validation',2426),('test',2428)]:
 with h5py.File(ROOT/'dataset'/f'eew_dataset_{split}.h5','r') as f:
  assert f.attrs['split']==split and f.attrs['sampling_rate_hz']==100
  for h in [1,2,3]:
   assert f[f'windows_{h}s_norm'].shape==(n,100+h*100)
   for key in ['labels','magnitude',f'classification_mask_{h}s',f'magnitude_mask_{h}s']:assert f[key].shape==(n,)
  print(split,n,'rows; 200/300/400 samples; 100 Hz; split/mask interface OK')
# Fail instead of overwriting an existing model.
assert not any((OUT/'models').iterdir()),'Models already exist; do not overwrite or retrain'
selected=[]
for h in [1,2,3]:
 seed_all(CFG['seed']);train=load_split('train',h);val=load_split('validation',h)
 train_loader=loader(train,True);train_eval=loader(train,False);val_loader=loader(val,False)
 center=float(train['mag'][train['reg']].mean());scale=float(train['mag'][train['reg']].std(unbiased=False));assert scale>0
 model=EarlyCNN().to(DEVICE);opt=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'])
 history=[];best=float('inf');stale=0;state=None;started=time.perf_counter()
 for e in range(1,CFG['max_epochs']+1):
  epoch(model,train_loader,center,scale,opt)
  tr=epoch(model,train_eval,center,scale);va=epoch(model,val_loader,center,scale)
  history.append({'epoch':e,'train':tr,'validation':va})
  if va['total']<best-CFG['min_delta']:
   best=va['total'];best_epoch=e;state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};stale=0
  else:stale+=1
  if e==1 or e%10==0:print(f'{h}s epoch {e}: train={tr["total"]:.4f}, val={va["total"]:.4f}',flush=True)
  if stale>=CFG['patience']:break
 assert state is not None
 ckpt={'state_dict':state,'horizon_seconds':h,'config':CFG,'target_center':center,'target_scale':scale,'best_epoch':best_epoch,'best_validation_loss':best,'history':history,'training_seconds':time.perf_counter()-started,'counts':{s:{'eligible':len(d['y']),'earthquakes':int(d['reg'].sum()),'excluded':d['excluded']} for s,d in [('train',train),('validation',val)]},'input_hashes':before,'architecture':'EarlyCNN: 1-16-32-48 strided Conv1d, mean/max pool, 96-32 shared, two heads','input_transform':'pointwise sign(x)*log1p(abs(x)); saved pre-P-normalized waveform only','environment':{'python':platform.python_version(),'torch':str(torch.__version__),'numpy':np.__version__,'device':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None},'parameter_count':sum(p.numel() for p in model.parameters())}
 path=OUT/'models'/f'cnn_{h}s_best.pt'
 with path.open('xb') as f:torch.save(ckpt,f)
 selected.append({'horizon':h,'best_epoch':best_epoch,'epochs_run':len(history),'val_loss':best,'seconds':round(ckpt['training_seconds'],2),'checkpoint':path.name})
 print('Selected:',selected[-1],flush=True)
assert all(sha(Path(p))==v for p,v in before.items()),'Protected input changed'
print('All three models selected without test metrics. Protected inputs unchanged.')


train 11324 rows; 200/300/400 samples; 100 Hz; split/mask interface OK
validation 2426 rows; 200/300/400 samples; 100 Hz; split/mask interface OK
test 2428 rows; 200/300/400 samples; 100 Hz; split/mask interface OK
1s epoch 1: train=1.0482, val=1.0492
1s epoch 10: train=0.9253, val=0.9415
1s epoch 20: train=0.8878, val=0.9427
Selected: {'horizon': 1, 'best_epoch': 16, 'epochs_run': 24, 'val_loss': 0.9337961459493991, 'seconds': 20.01, 'checkpoint': 'cnn_1s_best.pt'}
2s epoch 1: train=0.9271, val=0.9376
2s epoch 10: train=0.8120, val=0.8208
2s epoch 20: train=0.7788, val=0.8204
Selected: {'horizon': 2, 'best_epoch': 15, 'epochs_run': 23, 'val_loss': 0.8173246371440724, 'seconds': 17.82, 'checkpoint': 'cnn_2s_best.pt'}
3s epoch 1: train=0.9023, val=0.9257
3s epoch 10: train=0.7298, val=0.7641
3s epoch 20: train=0.6981, val=0.7570
Selected: {'horizon': 3, 'best_epoch': 14, 'epochs_run': 22, 'val_loss': 0.7538920510161912, 'seconds': 18.58, 'checkpoint': 'cnn_3s_best.pt'}
All three models 